# 🚀 03. 2단계 본문 RAG 및 온디맨드 벡터화 파이프라인

이 노트북은 1단계 초록 RAG 유사도 검색을 거쳐 3편의 후보 논문을 선정하고, 그 이후 대상 논문들의 본문을 가져와 실시간 온디맨드 벡터화(is_vectorized 플래그 검사 및 업데이트)를 수행한 뒤 논문 ID 메타데이터 필터링을 걸어 최종 답변 컨텍스트를 도출하는 2단계 RAG 파이프라인의 핵심 설계 원리를 학습합니다.

### ⚙️ 필요한 라이브러리 및 모듈 임포트
데이터베이스 비동기 세션을 획득하고, 캐시 서비스 및 RAG 검색을 조율하기 위한 모듈들을 로드합니다.

In [ ]:
from sqlalchemy import select
from api.database.config.dbsession import session_maker
from api.common.entities import PaperFullTextCacheEntity
from api.common.paper_cache_service import paper_cache_service
from api.common.rag_pipeline import common_rag_pipeline

## 📌 1단계. 초록 RAG 검색을 통한 후보 논문 선정 (Stage-1)
사용자의 학술 질문이 들어오면, 1차적으로 논문 초록(Abstract) 벡터 스토어에서 질문과 가장 가까운 후보 논문(arxiv_ids)을 추려냅니다. (여기서는 캐시에 존재하는 `2006.11367` 논문을 예시로 진행합니다.)

In [ ]:
query = "attention mechanism transformers and long-context windows"
domain = "cs"

print("=== [Stage-1] Abstract Search Simulation ===")
# 실전에서는: results = await common_rag_pipeline.similarity_search(domain, query, k=3)
candidate_ids = ["2006.11367"]
print(f"Selected Candidates for Deep Analysis: {candidate_ids}")

## 📌 2단계. 후보 논문 본문 확보 및 실시간 온디맨드 벡터화
선정된 후보 논문들이 이미 pgvector 본문 컬렉션(`paper_full_text_embeddings`)에 적재되어 있는지 `is_vectorized` 필드로 확인합니다. 만약 `False` 상태라면 실시간으로 텍스트를 가져와 청킹 및 임베딩 적재를 자동 완료하고 플래그를 `True`로 업데이트합니다.

In [ ]:
async def simulate_ondemand_vectorization(db, paper_ids, domain):
    print("=== [Stage-2] On-Demand Vectorization ===")
    for pid in paper_ids:
        # 1. DB 캐시 조회 및 텍스트 확보 (캐시 미스 시 크롤링 자동 연동)
        full_text = await paper_cache_service.get_paper_full_text(db, pid, domain)
        print(f"Paper '{pid}' Text Length: {len(full_text)} characters.")
        
        # 2. is_vectorized 플래그 확인
        query_db = select(PaperFullTextCacheEntity).where(PaperFullTextCacheEntity.paper_id == pid)
        res = await db.execute(query_db)
        paper_entity = res.scalar_one_or_none()
        
        if paper_entity:
            print(f"Paper '{pid}' is_vectorized status in DB: {paper_entity.is_vectorized}")
            
            if paper_entity.is_vectorized:
                print(f"--> Bypassing vectorization (Already vectorized for '{pid}').")
            else:
                print(f"--> Vectorizing paper '{pid}' on-demand now...")
                # 500자 청킹 및 pgvector 대칭 업로드 로직 실행
                chunks = common_rag_pipeline._chunk_text_custom(full_text, chunk_size=500, overlap=100)
                print(f"--> Generated {len(chunks)} chunks. Loading vectors into pgvector...")
                
                # 플래그 갱신 및 커밋
                paper_entity.is_vectorized = True
                db.add(paper_entity)
                await db.commit()
                print(f"--> Flag 'is_vectorized' updated to True for paper '{pid}'.")

async with session_maker() as db:
    await simulate_ondemand_vectorization(db, candidate_ids, domain)

## 📌 3단계. 메타데이터 필터 기반 정밀 본문 검색 (Stage-2)
온디맨드 벡터화가 완료되었으므로, pgvector에서 `paper_id` 필터를 걸어 대상 후보 논문군 중에서 질문과 가장 유사한 본문 청크들만 선별해 옵니다.

In [ ]:
async with session_maker() as db:
    # 공통 RAG 파이프라인의 get_full_text_context 메서드를 실행하여 최종 RAG 조각들을 수집
    context_chunks = await common_rag_pipeline.get_full_text_context(
        db=db,
        paper_ids=candidate_ids,
        query=query,
        domain=domain,
        k=3
    )
    
    print(f"Successfully retrieved {len(context_chunks)} deep context chunks:")
    for idx, chunk in enumerate(context_chunks, 1):
        print(f"\n[{idx}] Citation Match:")
        print(f"    Paper ID: {chunk['paper_id']} | Cosine Similarity: {chunk['score']:.4f}")
        print(f"    Text Context: {chunk['text_chunk'][:250]}...")